In [4]:
import pandas as pd
import numpy as np
from cmdstanpy import CmdStanModel
import geopandas as gpd
from libpysal import weights
from sklearn.preprocessing import LabelEncoder
import networkx as nx
import re
import os
import pickle
from scipy.sparse import diags

In [5]:
strat_frame = pd.read_csv("/mnt/e/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/feature_extraction/state_LP_cleaned_sf.csv")

In [6]:
strat_frame

,Unnamed: 0.1,Unnamed: 0,sex_idx,age_idx,eth_idx,income_idx,state_idx,vote_idx,education_idx,N_star.1,...,hh_income,vote2020,vote2022,state_abbreviation_upper,pct.2020.votes.R-D_zscore,State_Edu_LP,State_Edu_zscore,State_Trust_LP,Belief_opinion_on_the_number_of_immigrants_in_america_Symbol_LP,Belief_opinion_on_the_extent_to_which_this_individual_feels_it_is_justified_for_people_to_use_violence_to_pursue_their_political_goals_in_this_country_Symbol_LP
0,0,0,2,1,4,1,1,1,1,11.20473,...,"1.[min, 25000)",stay home,D,AK,0.398285,0.237451,-0.908230,0.796072,0.002266,-1.938357
1,1,1,2,1,4,1,1,2,1,0.08251,...,"1.[min, 25000)",R,D,AK,0.398285,0.237451,-0.908230,0.796072,0.002266,-1.938357
2,2,2,2,1,4,1,1,3,1,1.58995,...,"1.[min, 25000)",D,D,AK,0.398285,0.237451,-0.908230,0.796072,0.002266,-1.938357
3,3,3,2,1,4,1,1,4,1,0.01103,...,"1.[min, 25000)",L,D,AK,0.398285,0.237451,-0.908230,0.796072,0.002266,-1.938357
4,4,4,2,1,4,1,1,6,1,0.09892,...,"1.[min, 25000)",OTHER,D,AK,0.398285,0.237451,-0.908230,0.796072,0.002266,-1.938357
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
190839,190839,190839,1,6,1,5,51,3,2,183.35671,...,"5.[100000, max]",D,D,WY,1.790276,0.236046,-0.927473,1.515496,2.260323,-1.938357
190840,190840,190840,1,6,1,5,51,4,1,0.27655,...,"5.[100000, max]",L,D,WY,1.790276,0.236046,-0.927473,1.515496,2.260323,-1.938357
190841,190841,190841,1,6,1,5,51,4,2,0.94979,...,"5.[100000, max]",L,D,WY,1.790276,0.236046,-0.927473,1.515496,2.260323,-1.938357
190842,190842,190842,1,6,1,5,51,6,1,0.47236,...,"5.[100000, max]",OTHER,D,WY,1.790276,0.236046,-0.927473,1.515496,2.260323,-1.938357


In [7]:
strat_frame['N_star.median'].sum()

np.float64(10370279.246645)

In [8]:
strat_frame.columns.tolist()

['Unnamed: 0.1',
 'Unnamed: 0',
 'sex_idx',
 'age_idx',
 'eth_idx',
 'income_idx',
 'state_idx',
 'vote_idx',
 'education_idx',
 'N_star.1',
 'N_star.2',
 'N_star.3',
 'N_star.4',
 'N_star.5',
 'N_star.6',
 'N_star.7',
 'N_star.8',
 'N_star.9',
 'N_star.10',
 'N_star.11',
 'N_star.12',
 'N_star.13',
 'N_star.14',
 'N_star.15',
 'N_star.16',
 'N_star.17',
 'N_star.18',
 'N_star.19',
 'N_star.20',
 'N_star.21',
 'N_star.22',
 'N_star.23',
 'N_star.24',
 'N_star.25',
 'N_star.26',
 'N_star.27',
 'N_star.28',
 'N_star.29',
 'N_star.30',
 'N_star.31',
 'N_star.32',
 'N_star.33',
 'N_star.34',
 'N_star.35',
 'N_star.36',
 'N_star.37',
 'N_star.38',
 'N_star.39',
 'N_star.40',
 'N_star.41',
 'N_star.42',
 'N_star.43',
 'N_star.44',
 'N_star.45',
 'N_star.46',
 'N_star.47',
 'N_star.48',
 'N_star.49',
 'N_star.50',
 'N_star.51',
 'N_star.52',
 'N_star.53',
 'N_star.54',
 'N_star.55',
 'N_star.56',
 'N_star.57',
 'N_star.58',
 'N_star.59',
 'N_star.60',
 'N_star.61',
 'N_star.62',
 'N_star.63',

In [9]:
df = pd.read_pickle('/mnt/e/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/feature_extraction/preprocessed_cleaned_all_result.pkl')

In [10]:
df

,id,time_window_start,tweets_struct,name,username,description,location,public_metrics.followers_count,profile_image_url,clean...pool,...,Belief_opinion_on_whether_the_tone_of_political_discourse_in_the_u_s_has_become_more_factbased_Symbol_num,Belief_opinion_on_the_statement_elected_officials_using_heated_or_aggressive_language_to_talk_about_certain_people_or_groups_makes_violence_against_those_groups_more_likely_Symbol_cat,Belief_opinion_on_the_statement_elected_officials_using_heated_or_aggressive_language_to_talk_about_certain_people_or_groups_makes_violence_against_those_groups_more_likely_Symbol_num,Demo_sex_idx,Demo_age_idx,Demo_ethnicity_idx,Demo_income_idx,Demo_education_idx,Demo_past_vote_idx,state_cleaned_idx
0,1081532502966239232,2023-11-05 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,...,0,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1081532502966239232,2023-11-19 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,...,0,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1081532502966239232,2023-12-03 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,...,0,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1081532502966239232,2023-12-17 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,...,0,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1081532502966239232,2023-12-31 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,...,0,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23984,142827500,2024-08-11 00:00:00+00:00,"['12 weeks to election', {'text': '@maxies_bac...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,...,2,dib1,1,1.0,2.0,1.0,4.0,2.0,3.0,15.0
23985,142827500,2024-08-25 00:00:00+00:00,"['10 weeks to election', {'text': '@foldyrhand...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,...,2,dib1,1,1.0,2.0,1.0,3.0,2.0,3.0,15.0
23987,142827500,2024-09-22 00:00:00+00:00,"['6 weeks to election', {'text': '@kevin__fm S...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,...,2,dib1,1,1.0,2.0,1.0,3.0,2.0,3.0,15.0
23988,142827500,2024-10-06 00:00:00+00:00,"['4 weeks to election', {'text': '@cherryrepre...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,...,2,dib1,1,1.0,2.0,1.0,3.0,2.0,1.0,15.0


In [11]:
print(df.columns.to_list())

['id', 'time_window_start', 'tweets_struct', 'name', 'username', 'description', 'location', 'public_metrics.followers_count', 'profile_image_url', 'clean...pool', 'is_empty_window', 'window_weeks_to_election', 'profile_image_description', 'profile_image_main_info', 'Model', 'Belief_Raw', 'Belief_Cleaned', 'Demo_Raw', 'Demo_Cleaned', 'Belief_opinion_on_the_number_of_immigrants_in_america_Category', 'Belief_opinion_on_the_number_of_immigrants_in_america_Explanation', 'Belief_opinion_on_the_number_of_immigrants_in_america_Symbol', 'Belief_opinion_on_the_number_of_immigrants_in_america_Speculation', 'Belief_opinion_on_the_statement_immigrants_increase_crime_rates_Category', 'Belief_opinion_on_the_statement_immigrants_increase_crime_rates_Explanation', 'Belief_opinion_on_the_statement_immigrants_increase_crime_rates_Symbol', 'Belief_opinion_on_the_statement_immigrants_increase_crime_rates_Speculation', 'Belief_opinion_on_the_statement_immigrants_take_jobs_away_from_people_who_were_born_in_t

In [12]:
belief_col = 'Belief_opinion_on_the_extent_to_which_this_individual_feels_it_is_justified_for_people_to_use_violence_to_pursue_their_political_goals_in_this_country_Symbol'

In [13]:
predictor_cols = [
    "Demo_sex_Symbol",
    "Demo_age_Symbol",
    "Demo_ethnicity_Symbol",
    "Demo_income_Symbol",
    "Demo_education_Symbol",
    "Demo_past_vote_Symbol",
    "state_cleaned"
]
df = df.dropna(subset=[belief_col] + predictor_cols)

In [14]:
df

,id,time_window_start,tweets_struct,name,username,description,location,public_metrics.followers_count,profile_image_url,clean...pool,...,Belief_opinion_on_whether_the_tone_of_political_discourse_in_the_u_s_has_become_more_factbased_Symbol_num,Belief_opinion_on_the_statement_elected_officials_using_heated_or_aggressive_language_to_talk_about_certain_people_or_groups_makes_violence_against_those_groups_more_likely_Symbol_cat,Belief_opinion_on_the_statement_elected_officials_using_heated_or_aggressive_language_to_talk_about_certain_people_or_groups_makes_violence_against_those_groups_more_likely_Symbol_num,Demo_sex_idx,Demo_age_idx,Demo_ethnicity_idx,Demo_income_idx,Demo_education_idx,Demo_past_vote_idx,state_cleaned_idx
40,1542629834953719810,2024-05-19 00:00:00+00:00,"['24 weeks to election', {'text': '@The_Trump_...",Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,2,dib1,1,2.0,3.0,1.0,3.0,1.0,2.0,36.0
41,1542629834953719810,2024-06-16 00:00:00+00:00,"['20 weeks to election', {'text': '🤣 https://t...",Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,2,dib1,1,2.0,3.0,1.0,3.0,1.0,2.0,36.0
43,1542629834953719810,2024-07-14 00:00:00+00:00,"['16 weeks to election', {'text': '🤣🤣🤣 https:/...",Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,2,dib3,3,2.0,3.0,1.0,3.0,1.0,2.0,36.0
44,1542629834953719810,2024-07-28 00:00:00+00:00,"['14 weeks to election', {'text': '@victormarx...",Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,2,dib1,1,2.0,3.0,1.0,2.0,1.0,3.0,36.0
45,1542629834953719810,2024-08-11 00:00:00+00:00,"['12 weeks to election', {'text': '🤣🥰 https://...",Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,2,dib1,1,2.0,3.0,1.0,3.0,1.0,2.0,36.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23984,142827500,2024-08-11 00:00:00+00:00,"['12 weeks to election', {'text': '@maxies_bac...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,...,2,dib1,1,1.0,2.0,1.0,4.0,2.0,3.0,15.0
23985,142827500,2024-08-25 00:00:00+00:00,"['10 weeks to election', {'text': '@foldyrhand...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,...,2,dib1,1,1.0,2.0,1.0,3.0,2.0,3.0,15.0
23987,142827500,2024-09-22 00:00:00+00:00,"['6 weeks to election', {'text': '@kevin__fm S...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,...,2,dib1,1,1.0,2.0,1.0,3.0,2.0,3.0,15.0
23988,142827500,2024-10-06 00:00:00+00:00,"['4 weeks to election', {'text': '@cherryrepre...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,...,2,dib1,1,1.0,2.0,1.0,3.0,2.0,1.0,15.0


In [15]:
# prepare spatial structure for ICAR
def edges_from_adjacency(W):
    all_edges = []
    n = W.shape[0]
    
    for i in range(n):
        neighbors = np.where(W[i, :] != 0)[0]
        for j in neighbors:
            if i < j:  
                all_edges.append({
                    'node1': i + 1,  
                    'node2': j + 1,
                    'weight': W[i, j]
                })
    
    return pd.DataFrame(all_edges)


def scale_c(C):
    N = C.shape[0]
    
    if hasattr(C, 'toarray'):
        C = C.toarray()
    C = np.asarray(C)
    
    row_sums = C.sum(axis=1).ravel()
    Q = diags(row_sums) - C
    
    Q_dense = Q.toarray() if hasattr(Q, 'toarray') else np.asarray(Q)
    max_diag = np.max(np.diag(Q_dense))
    jitter = max_diag * np.sqrt(np.finfo(float).eps)
    Q_pert = Q_dense + np.diag(np.ones(N) * jitter)
    
    Q_inv = np.linalg.pinv(Q_pert)
    
    variances = np.diag(Q_inv)
    scaling_factor = np.exp(np.mean(np.log(variances)))
    
    return scaling_factor


def prep_icar_data_from_weights(w, N_states):
    C = np.zeros((N_states, N_states))
    for i in range(N_states):
        for j in w.neighbors[i]:
            C[i, j] = 1

    E = edges_from_adjacency(C)
    
    G = nx.Graph()
    G.add_nodes_from(range(N_states))
    for _, row in E.iterrows():
        G.add_edge(int(row['node1']) - 1, int(row['node2']) - 1)
    
    components = list(nx.connected_components(G))
    k = len(components)
    
    comp_id = np.zeros(N_states, dtype=int)
    for idx, component in enumerate(components):
        for node in component:
            comp_id[int(node)] = idx
    
    group_size = np.array([len(comp) for comp in components])
    
    group_idx = comp_id + 1

    
    return {
        'k': int(k),
        'group_size': group_size.tolist(),
        'N_edges': len(E),
        'node1': E['node1'].values.tolist(),
        'node2': E['node2'].values.tolist(),
        'group_idx': group_idx.tolist()
    }


def moran_coefficient(x, W, warn=True):
    row_sums = W.sum(axis=1)
    zero_idx = np.where(row_sums == 0)[0]
    
    if len(zero_idx) > 0:
        if warn:
            print(f"{len(zero_idx)} observations with no neighbors. Dropping from calculation.")
        keep_idx = row_sums > 0
        x = x[keep_idx]
        W = W[np.ix_(keep_idx, keep_idx)]
    
    n = len(x)
    xbar = np.mean(x)
    z = x - xbar
    z_tilde = W @ z
    A = np.sum(W)
    mc = (n / A) * (z @ z_tilde) / (z @ z)
    
    return np.round(mc, 3)

# Build spatial structure
shp = gpd.read_file("/mnt/e/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/tl_2025_us_state/tl_2025_us_state.shp")
shp["STATE"] = shp["STUSPS"].str.upper()

shp = shp[~shp["STATE"].isin(["PR", "GU", "VI", "MP", "AS"])]

shp = shp.sort_values("STATE").reset_index(drop=True)

state_order = shp["STATE"].tolist()
N_states = len(state_order)

print(f"Shapefile states: {N_states}")

w = weights.Queen.from_dataframe(shp)

icar_data = prep_icar_data_from_weights(w, N_states)

print(f"\nICAR structure:")
print(f"  Connected components: {icar_data['k']}")
print(f"  Component sizes: {icar_data['group_size']}")
print(f"  Number of edges: {icar_data['N_edges']}")



Shapefile states: 51


/tmp/ipykernel_13086/4196857512.py:111: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w = weights.Queen.from_dataframe(shp)



ICAR structure:
  Connected components: 3
  Component sizes: [1, 49, 1]
  Number of edges: 112


/home/yuyue/stan_env/lib/python3.12/site-packages/libpysal/weights/contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 3 disconnected components.
 There are 2 islands with ids: 0, 11.
  W.__init__(self, neighbors, ids=ids, **kw)


In [16]:
shp.head()

,REGION,DIVISION,STATEFP,STATENS,GEOID,GEOIDFQ,STUSPS,NAME,LSAD,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry,STATE
0,4,9,02,01785533,02,0400000US02,AK,Alaska,00,G4000,A,1479893380150,244326118163,+63.3473560,-152.8397334,"MULTIPOLYGON (((-167.55823 60.22436, -167.5567...",AK
1,3,6,01,01779775,01,0400000US01,AL,Alabama,00,G4000,A,131186429591,4580729056,+32.7395785,-086.8434469,"POLYGON ((-85.12733 31.76256, -85.12753 31.762...",AL
2,3,7,05,00068085,05,0400000US05,AR,Arkansas,00,G4000,A,134658642291,3122591253,+34.8955256,-092.4446262,"POLYGON ((-90.95577 34.11871, -90.95451 34.117...",AR
3,4,8,04,01779777,04,0400000US04,AZ,Arizona,00,G4000,A,294366238828,853868947,+34.2039362,-111.6063449,"POLYGON ((-110.75069 37.00301, -110.74193 37.0...",AZ
4,4,9,06,01779778,06,0400000US06,CA,California,00,G4000,A,403676564152,20288505023,+37.1551773,-119.5434183,"MULTIPOLYGON (((-119.99987 41.18397, -119.9998...",CA


In [17]:
strat_frame[f'{belief_col}_LP']

0        -1.938357
1        -1.938357
2        -1.938357
3        -1.938357
4        -1.938357
            ...   
190839   -1.938357
190840   -1.938357
190841   -1.938357
190842   -1.938357
190843   -1.938357
Name: Belief_opinion_on_the_extent_to_which_this_individual_feels_it_is_justified_for_people_to_use_violence_to_pursue_their_political_goals_in_this_country_Symbol_LP, Length: 190844, dtype: float64

In [ ]:
# state level fixed effects
LP_cols = [
    'pct.2020.votes.R-D_zscore', 'pct.2020.votes.stay.home_z_score','State_Edu_zscore', 'State_Trust_LP',f'{belief_col}_LP'
]
LP_state = (
    strat_frame
    .assign(STATE=strat_frame["state_abbreviation"].str.upper())
    .groupby("STATE")[LP_cols]
    .mean()
)
LP_state = LP_state.reindex(state_order)

LP_mat = LP_state.values
P = LP_mat.shape[1]
print(LP_mat.shape)

(51, 5)


In [19]:
print(LP_mat)

[[ 3.98285422e-01 -1.54946990e-01 -9.08229582e-01  7.96071521e-01
  -1.93835710e+00]
 [ 1.04690352e+00  9.26956911e-01 -1.07237637e+00  1.86548581e+00
   8.75065729e-01]
 [ 1.13783431e+00  1.87393581e+00 -1.38331043e+00  1.83923655e+00
   9.72080309e-01]
 [-3.84615563e-02 -1.35498834e-01 -2.90815875e-02 -1.24791401e-01
  -2.38974183e-01]
 [-1.25372120e+00 -6.12608358e-02  7.09289140e-01 -1.40207633e-01
   8.10482135e-01]
 [-5.90117107e-01 -1.72932917e+00  1.76367409e+00 -1.07443129e+00
   7.51978480e-01]
 [-8.70895954e-01 -3.63993639e-01  1.31897850e+00  3.82402630e-01
  -1.52579507e-02]
 [-3.67919160e+00  3.05218130e-01  3.88149541e+00 -5.88819983e-01
   3.82716734e-01]
 [-8.24336502e-01 -3.95519905e-01  3.00571267e-01  2.19705541e+00
  -1.49413244e+00]
 [ 1.15984037e-01 -8.03509506e-01  1.35215925e-01  6.99342222e-01
   6.95383509e-02]
 [-3.53816519e-02 -1.24214759e-01  2.22020430e-02  1.12938271e+00
   2.60038413e-01]
 [-1.25504545e+00  1.97166449e+00  9.01199576e-02  6.22417414e-01

In [20]:
LP_df = pd.DataFrame(
    LP_mat,
    index=[s.upper() for s in state_order],
    columns=LP_cols
)

LP_df.head()

,pct.2020.votes.R-D_zscore,LP_pct.2020.votes.stay.home,State_Edu_zscore,State_Trust_LP,Belief_opinion_on_the_extent_to_which_this_individual_feels_it_is_justified_for_people_to_use_violence_to_pursue_their_political_goals_in_this_country_Symbol_LP
AK,0.398285,-0.154947,-0.908230,0.796072,-1.938357
AL,1.046904,0.926957,-1.072376,1.865486,0.875066
AR,1.137834,1.873936,-1.383310,1.839237,0.972080
AZ,-0.038462,-0.135499,-0.029082,-0.124791,-0.238974
CA,-1.253721,-0.061261,0.709289,-0.140208,0.810482


In [21]:
print(df["Demo_age_idx"].unique())

[3. 4. 5. 2. 6. 1.]


In [22]:
# Stan code
stan_code = '''
functions {
  real icar_normal_lpdf(vector phi, int N_edges,
                        array[] int node1, array[] int node2) {
    return -0.5 * dot_self(phi[node1] - phi[node2]);
  }
}

data {
  int<lower=1> N;                   
  int<lower=2> K;                 
  array[N] int<lower=1, upper=K> y;

  int<lower=1> J_sex;
  int<lower=1> J_age;
  int<lower=1> J_eth;
  int<lower=1> J_income;
  int<lower=1> J_education;
  int<lower=1> J_vote;
  int<lower=1> N_states;

  array[N] int<lower=1, upper=J_sex> sex_idx;
  array[N] int<lower=1, upper=J_age> age_idx;
  array[N] int<lower=1, upper=J_eth> eth_idx;
  array[N] int<lower=1, upper=J_income> income_idx;
  array[N] int<lower=1, upper=J_education> education_idx;
  array[N] int<lower=1, upper=J_vote> vote_idx;
  array[N] int<lower=1, upper=N_states> state_idx;

  // ICAR structure
  int<lower=1> N_edges;
  array[N_edges] int node1;
  array[N_edges] int node2;

  int<lower=1> k;                           // number of connected components
  array[k] int group_size;                  // size of each connected component
  array[N_states] int group_idx;           // component assignment for each state

  // state level fixed effects
  int<lower=1> P;                           // number of state-level predictors
  matrix[N_states, P] LP;                  // state-level predictor matrix
}

parameters {
  ordered[K-1] kappa;

  sum_to_zero_vector[J_sex] eta_sex; // use build in sum_to_zero_vector instead of hard coding it
  sum_to_zero_vector[J_age] eta_age;             
  sum_to_zero_vector[J_eth] eta_eth;
  sum_to_zero_vector[J_income] eta_income; 
  sum_to_zero_vector[J_education] eta_education;     
  sum_to_zero_vector[J_vote] eta_vote;
  vector[N_states] eta_state_raw;
  vector[P] beta_LP;

  real<lower=0> sigma_sex;
  real<lower=0> sigma_age;
  real<lower=0> sigma_eth;
  real<lower=0> sigma_income;
  real<lower=0> sigma_education;
  real<lower=0> sigma_vote;
  real<lower=0> sigma_state;
}

transformed parameters {
  vector[J_sex] eta_sex_star = eta_sex * sigma_sex ;
  vector[J_age] eta_age_star = eta_age * sigma_age;
  vector[J_eth] eta_eth_star = eta_eth * sigma_eth ;
  vector[J_income] eta_income_star = eta_income * sigma_income;
  vector[J_education] eta_education_star = eta_education * sigma_education;
  vector[J_vote] eta_vote_star = eta_vote * sigma_vote;
  vector[N_states] mu_state_LP = LP * beta_LP; 
  vector[N_states] eta_state = sigma_state * eta_state_raw;
}

model {
  // Priors
  kappa ~ normal(0, 3);
  beta_LP ~ normal(0, 1);

   // prior on the constrained vector (Stan handles the internal K-1 representation)
  eta_sex ~ normal(0, 1);
  eta_eth ~ normal(0, 1);
  eta_vote ~ normal(0, 1);
  eta_education ~ normal(0, 1);
  sigma_sex ~ normal(0, 1);
  sigma_age ~ normal(0, 1);
  sigma_eth ~ normal(0, 1);
  sigma_income ~ normal(0, 1);
  sigma_education ~ normal(0, 1);
  sigma_vote ~ normal(0, 1);
  sigma_state ~ normal(0, 1);

  for (i in 2:J_age) {
    eta_age[i] ~ normal(eta_age[i-1], 1);
  }
  for (i in 2:J_income) {
    eta_income[i] ~ normal(eta_income[i-1], 1);
  }

  // ICAR prior on raw spatial effects
  target += icar_normal_lpdf(eta_state_raw | N_edges, node1, node2);
  
  // Sum-to-zero constraint per connected component (island can have free prior)
  for (j in 1:k) {
  if (group_size[j] > 1) {
    // Soft sum-to-zero constraint for proper ICAR within this component
    real component_sum = 0;
    for (i in 1:N_states) {
      if (group_idx[i] == j) {
        component_sum += eta_state_raw[i];
      }
    }
    target += normal_lpdf(component_sum | 0, 0.001 * group_size[j]);
  } else {
    // Island component: give the singleton node a proper iid prior
    for (i in 1:N_states) {
      if (group_idx[i] == j) {
        eta_state_raw[i] ~ normal(0, 1);
      }
    }
  }
}

  // Ordered logistic
  for (i in 1:N) {
    real mu = eta_sex_star[sex_idx[i]]
              + eta_age_star[age_idx[i]]
              + eta_eth_star[eth_idx[i]]
              + eta_income_star[income_idx[i]]
              + eta_vote_star[vote_idx[i]]
              + eta_education_star[education_idx[i]]
              + eta_state[state_idx[i]]
              + mu_state_LP[state_idx[i]];
    y[i] ~ ordered_logistic(mu, kappa);
  }
}

generated quantities {
  vector[N] mu;
  vector[N] log_lik;
  
  for (i in 1:N) {
    mu[i] = eta_sex_star[sex_idx[i]]
            + eta_age_star[age_idx[i]]
            + eta_eth_star[eth_idx[i]]
            + eta_income_star[income_idx[i]]
            + eta_vote_star[vote_idx[i]]
            + eta_education_star[education_idx[i]]
            + eta_state[state_idx[i]]
            + mu_state_LP[state_idx[i]];
    log_lik[i] = ordered_logistic_lpmf(y[i] | mu[i], kappa);
  }
}
'''


# Compile and Sample

with open('mrp.stan', 'w') as f:
    f.write(stan_code)

model = CmdStanModel(stan_file='mrp.stan')


22:24:41 - cmdstanpy - INFO - compiling stan file /tmp/tmp9k80r5py/tmpjlb87may.stan to exe file /mnt/c/Users/yuyue/AppData/Local/Programs/Microsoft VS Code/mrp
22:24:58 - cmdstanpy - INFO - compiled model executable: /mnt/c/Users/yuyue/AppData/Local/Programs/Microsoft VS Code/mrp


In [23]:
time_windows = sorted(df['time_window_start'].unique())
print(f"Time windows found: {time_windows}")

for idx,window in enumerate(time_windows):
    df_window = df[df['time_window_start'] == window].copy()
    print(len(df_window))

Time windows found: ['2023-11-05 00:00:00+00:00', '2023-11-19 00:00:00+00:00', '2023-12-03 00:00:00+00:00', '2023-12-17 00:00:00+00:00', '2023-12-31 00:00:00+00:00', '2024-01-14 00:00:00+00:00', '2024-01-28 00:00:00+00:00', '2024-02-11 00:00:00+00:00', '2024-02-25 00:00:00+00:00', '2024-03-10 00:00:00+00:00', '2024-03-24 00:00:00+00:00', '2024-04-07 00:00:00+00:00', '2024-04-21 00:00:00+00:00', '2024-05-05 00:00:00+00:00', '2024-05-19 00:00:00+00:00', '2024-06-02 00:00:00+00:00', '2024-06-16 00:00:00+00:00', '2024-06-30 00:00:00+00:00', '2024-07-14 00:00:00+00:00', '2024-07-28 00:00:00+00:00', '2024-08-11 00:00:00+00:00', '2024-08-25 00:00:00+00:00', '2024-09-08 00:00:00+00:00', '2024-09-22 00:00:00+00:00', '2024-10-06 00:00:00+00:00', '2024-10-20 00:00:00+00:00']
165
166
162
177
184
195
203
209
220
235
245
251
259
269
311
307
329
351
397
417
439
480
541
421
300
123


In [24]:
time_windows = sorted(df['time_window_start'].unique())
print(f"Time windows found: {time_windows}")
print(len(time_windows))

results_by_window = {}

for idx,window in enumerate(time_windows):
    # Filter data for this window
    print(f"[{idx+1}/{len(time_windows)}] Processing window: {window}")
    df_window = df[df['time_window_start'] == window].copy()
    print(f"  Observations: {len(df_window)}")
    print(f"  Belief categories: {sorted(df_window[belief_col+'_num'].unique())}")
    stan_data = {
        "N": len(df_window),
        "K": df[belief_col+"_cat"].cat.categories.size,
        "y": df_window[belief_col+"_num"].astype(int).tolist(),
        
        # Predictor dimensions
        "J_sex": strat_frame['sex_idx'].nunique(),
        "J_age": strat_frame['age_idx'].nunique(),
        "J_eth": strat_frame['eth_idx'].nunique(),
        "J_income": strat_frame['income_idx'].nunique(),
        "J_vote": strat_frame['vote_idx'].nunique(),
        "J_education": strat_frame['education_idx'].nunique(),
        "N_states": N_states,
        

        # Predictor indices
        "sex_idx": df_window['Demo_sex_idx'].astype(int).tolist(),
        "age_idx": df_window['Demo_age_idx'].astype(int).tolist(),
        "eth_idx": df_window['Demo_ethnicity_idx'].astype(int).tolist(),
        "income_idx": df_window['Demo_income_idx'].astype(int).tolist(),
        "vote_idx": df_window['Demo_past_vote_idx'].astype(int).tolist(),
        "education_idx": df_window['Demo_education_idx'].astype(int).tolist(),
        "state_idx": df_window['state_cleaned_idx'].astype(int).tolist(),

        # ICAR spatial structure
        **icar_data,

        # state level fix effect
        "P": LP_mat.shape[1],
        "LP": LP_mat.tolist(),
    }
    
    fit = model.sample(
        data=stan_data,
        chains=8,
        parallel_chains=8,
        iter_warmup=2000,
        iter_sampling=1000,
        adapt_delta=0.995,
        max_treedepth=13,
        thin = 32,
        sig_figs=18,
        show_console=True
    )

    print(fit.summary())

    base_dir = f"/mnt/e/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/{belief_col}"
    model_dir = f"{base_dir}/{belief_col}_{window}_stan_output"

    os.makedirs(model_dir, exist_ok=True)

    summary_df = fit.summary()

    summary_df.to_csv(f"{model_dir}/mrp_summary.csv", index=True)

    with open(f"{model_dir}/fit.pkl", "wb") as f:
        pickle.dump(fit, f)
        

22:24:58 - cmdstanpy - INFO - Chain [1] start processing
22:24:58 - cmdstanpy - INFO - Chain [2] start processing
22:24:58 - cmdstanpy - INFO - Chain [3] start processing
22:24:58 - cmdstanpy - INFO - Chain [4] start processing
22:24:58 - cmdstanpy - INFO - Chain [5] start processing
22:24:58 - cmdstanpy - INFO - Chain [6] start processing
22:24:58 - cmdstanpy - INFO - Chain [7] start processing
22:24:58 - cmdstanpy - INFO - Chain [8] start processing


Time windows found: ['2023-11-05 00:00:00+00:00', '2023-11-19 00:00:00+00:00', '2023-12-03 00:00:00+00:00', '2023-12-17 00:00:00+00:00', '2023-12-31 00:00:00+00:00', '2024-01-14 00:00:00+00:00', '2024-01-28 00:00:00+00:00', '2024-02-11 00:00:00+00:00', '2024-02-25 00:00:00+00:00', '2024-03-10 00:00:00+00:00', '2024-03-24 00:00:00+00:00', '2024-04-07 00:00:00+00:00', '2024-04-21 00:00:00+00:00', '2024-05-05 00:00:00+00:00', '2024-05-19 00:00:00+00:00', '2024-06-02 00:00:00+00:00', '2024-06-16 00:00:00+00:00', '2024-06-30 00:00:00+00:00', '2024-07-14 00:00:00+00:00', '2024-07-28 00:00:00+00:00', '2024-08-11 00:00:00+00:00', '2024-08-25 00:00:00+00:00', '2024-09-08 00:00:00+00:00', '2024-09-22 00:00:00+00:00', '2024-10-06 00:00:00+00:00', '2024-10-20 00:00:00+00:00']
26
[1/26] Processing window: 2023-11-05 00:00:00+00:00
  Observations: 165
  Belief categories: [np.int8(1), np.int8(3), np.int8(4), np.int8(5)]
Chain [3] method = sample (Default)
Chain [6] method = sample (Default)
Chain [6

22:27:46 - cmdstanpy - INFO - Chain [1] done processing


Chain [6] Iteration: 2500 / 3000 [ 83%]  (Sampling)
Chain [1] Iteration: 3000 / 3000 [100%]  (Sampling)
Chain [1] 
Chain [1] Elapsed Time: 117.154 seconds (Warm-up)
Chain [1] 54.691 seconds (Sampling)
Chain [1] 171.845 seconds (Total)
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [4] Iteration: 2500 / 3000 [ 83%]  (Sampling)
Chain [7] Iteration: 2800 / 3000 [ 93%]  (Sampling)
Chain [8] Iteration: 2800 / 3000 [ 93%]  (Sampling)
Chain [3] Iteration: 2600 / 3000 [ 86%]  (Sampling)
Chain [5] Iteration: 2300 / 3000 [ 76%]  (Sampling)
Chain [2] Iteration: 2700 / 3000 [ 90%]  (Sampling)
Chain [6] Iteration: 2600 / 3000 [ 86%]  (Sampling)
Chain [8] Iteration: 2900 / 3000 [ 96%]  (Sampling)
Chain [3] Iteration: 2700 / 3000 [ 90%]  (Sampling)
Chain [7] Iteration: 2900 / 3000 [ 96%]  (Sa

22:27:58 - cmdstanpy - INFO - Chain [8] done processing
22:27:59 - cmdstanpy - INFO - Chain [7] done processing


Chain [8] Iteration: 3000 / 3000 [100%]  (Sampling)
Chain [8] 
Chain [8] Elapsed Time: 121.238 seconds (Warm-up)
Chain [8] 63.414 seconds (Sampling)
Chain [8] 184.652 seconds (Total)
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [8] 
Chain [7] Iteration: 3000 / 3000 [100%]  (Sampling)
Chain [3] Iteration: 2800 / 3000 [ 93%]  (Sampling)
Chain [7] 
Chain [7] Elapsed Time: 122.179 seconds (Warm-up)
Chain [7] 62.571 seconds (Sampling)
Chain [7] 184.75 seconds (Total)
Chain [7] 


... (452 outputs left out)

01:11:34 - cmdstanpy - INFO - Chain [1] done processing


Chain [1] Iteration: 3000 / 3000 [100%]  (Sampling)
Chain [1] 
Chain [1] Elapsed Time: 110.986 seconds (Warm-up)
Chain [1] 47.415 seconds (Sampling)
Chain [1] 158.401 seconds (Total)
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Cha

01:11:53 - cmdstanpy - INFO - Chain [7] done processing
01:11:53 - cmdstanpy - WARNING - Non-fatal error during sampling:
Exception: ordered_logistic: Cut-points is not a valid ordered vector. The element at 2 is -760.691, but should be greater than the previous element, -760.691 (in 'mrp.stan', line 135, column 4 to column 39)
Exception: ordered_logistic: Cut-points is not a valid ordered vector. The element at 3 is -52.8639, but should be greater than the previous element, -52.8639 (in 'mrp.stan', line 135, column 4 to column 39)
	Exception: ordered_logistic: Cut-points is not a valid ordered vector. The element at 3 is -54.6161, but should be greater than the previous element, -54.6161 (in 'mrp.stan', line 135, column 4 to column 39)
	Exception: ordered_logistic: Cut-points is not a valid ordered vector. The element at 2 is -5525.16, but should be greater than the previous element, -5525.16 (in 'mrp.stan', line 135, column 4 to column 39)
	Exception: ordered_logistic: Cut-points is 

Chain [7] Iteration: 3000 / 3000 [100%]  (Sampling)
Chain [7] 
Chain [7] Elapsed Time: 106.138 seconds (Warm-up)
Chain [7] 72.558 seconds (Sampling)
Chain [7] 178.696 seconds (Total)
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Cha

# Get Predictions

In [25]:
base_dir = f"/mnt/e/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/{belief_col}"
model_dir = f"{base_dir}/{belief_col}_{window}_stan_output"

In [29]:
import pandas as pd

def generate_cell_level_predictions(
    fit,
    strat_frame,
    belief_col,
    batch_size=10000,
    n_keep_draws=250
):

    # Extract posterior samples
    kappa = fit.stan_variable('kappa')  # 2D array (draws x K-1)
    eta_sex_star = fit.stan_variable('eta_sex_star')
    eta_age_star = fit.stan_variable('eta_age_star')
    eta_eth_star = fit.stan_variable('eta_eth_star')
    eta_vote_star = fit.stan_variable('eta_vote_star')
    eta_income_star = fit.stan_variable('eta_income_star')
    eta_education_star = fit.stan_variable('eta_education_star')
    eta_state = fit.stan_variable('eta_state')
    mu_state_LP = fit.stan_variable('mu_state_LP')

    n_draws_total = kappa.shape[0]
    n_cells = len(strat_frame)
    K = kappa.shape[1] + 1

    # Randomly select indices for draw subsampling
    draw_indices = np.random.choice(
        n_draws_total,
        size=min(n_keep_draws, n_draws_total),
        replace=False
    )
    draw_indices = np.sort(draw_indices)
    n_draws = len(draw_indices)

    # Subset posterior samples to keep only selected draws
    kappa = kappa[draw_indices, :]
    eta_sex_star = eta_sex_star[draw_indices, :]
    eta_age_star = eta_age_star[draw_indices, :]
    eta_eth_star = eta_eth_star[draw_indices, :]
    eta_vote_star = eta_vote_star[draw_indices, :]
    eta_income_star = eta_income_star[draw_indices, :]
    eta_education_star = eta_education_star[draw_indices, :]
    eta_state = eta_state[draw_indices, :]
    mu_state_LP = mu_state_LP[draw_indices, :]

    print(f"Subsampling draws: using {n_draws} out of {n_draws_total} posterior draws")

    # Initialize arrays
    pred_mean = np.zeros(n_cells)
    pred_median = np.zeros(n_cells)
    pred_sd = np.zeros(n_cells)

    prob_cats_mean = np.zeros((n_cells, K))
    prob_cats_median = np.zeros((n_cells, K))
    prob_cats_sd = np.zeros((n_cells, K))

    pred_lower_95 = np.zeros(n_cells)
    pred_upper_95 = np.zeros(n_cells)
    pred_lower_90 = np.zeros(n_cells)
    pred_upper_90 = np.zeros(n_cells)
    pred_lower_80 = np.zeros(n_cells)
    pred_upper_80 = np.zeros(n_cells)
    pred_lower_68 = np.zeros(n_cells)
    pred_upper_68 = np.zeros(n_cells)

    prob_cats_lower_95 = np.zeros((n_cells, K))
    prob_cats_upper_95 = np.zeros((n_cells, K))
    prob_cats_lower_90 = np.zeros((n_cells, K))
    prob_cats_upper_90 = np.zeros((n_cells, K))
    prob_cats_lower_80 = np.zeros((n_cells, K))
    prob_cats_upper_80 = np.zeros((n_cells, K))
    prob_cats_lower_68 = np.zeros((n_cells, K))
    prob_cats_upper_68 = np.zeros((n_cells, K))

    all_draws = np.zeros((n_cells, n_draws))  # expected value draws
    prob_all_draws = np.zeros((n_cells, n_draws, K))  # category probability draws

    n_batches = int(np.ceil(n_cells / batch_size))
    print(f"\nGenerating predictions for {n_cells:,} cells in {n_batches} batches of size {batch_size}")

    for batch_idx in range(n_batches):
        start_idx = batch_idx * batch_size
        end_idx = min((batch_idx + 1) * batch_size, n_cells)
        batch_n = end_idx - start_idx
        batch_data = strat_frame.iloc[start_idx:end_idx]

        mu_batch = np.zeros((batch_n, n_draws))

        for i, (_, row) in enumerate(batch_data.iterrows()):
            mu_batch[i, :] = (
                eta_sex_star[:, int(row['sex_idx']) - 1]
                + eta_age_star[:, int(row['age_idx']) - 1]
                + eta_vote_star[:, int(row['vote_idx']) - 1]
                + eta_eth_star[:, int(row['eth_idx']) - 1]
                + eta_income_star[:, int(row['income_idx']) - 1]
                + eta_education_star[:, int(row['education_idx']) - 1]
                + eta_state[:, int(row['state_idx']) - 1]
                + mu_state_LP[:, int(row['state_idx']) - 1]
            )

        # calculate probabilities for each category
        prob_batch = np.zeros((batch_n, n_draws, K))
        for k in range(K):
            if k == 0:
                prob_batch[:, :, k] = 1 / (1 + np.exp(-(kappa[:, 0] - mu_batch)))
            elif k == K - 1:
                prob_batch[:, :, k] = 1 - 1 / (1 + np.exp(-(kappa[:, K - 2] - mu_batch)))
            else:
                prob_batch[:, :, k] = (
                    1 / (1 + np.exp(-(kappa[:, k] - mu_batch)))
                    - 1 / (1 + np.exp(-(kappa[:, k - 1] - mu_batch)))
                )

        # calculate expected value
        categories = np.arange(1, K + 1)
        expected_y_batch = np.sum(
            prob_batch * categories[np.newaxis, np.newaxis, :],
            axis=2
        )

        # summary stats for expected value
        pred_mean[start_idx:end_idx] = np.mean(expected_y_batch, axis=1)
        pred_median[start_idx:end_idx] = np.median(expected_y_batch, axis=1)
        pred_lower_95[start_idx:end_idx] = np.percentile(expected_y_batch, 2.5, axis=1)
        pred_upper_95[start_idx:end_idx] = np.percentile(expected_y_batch, 97.5, axis=1)
        pred_lower_90[start_idx:end_idx] = np.percentile(expected_y_batch, 5, axis=1)
        pred_upper_90[start_idx:end_idx] = np.percentile(expected_y_batch, 95, axis=1)
        pred_lower_80[start_idx:end_idx] = np.percentile(expected_y_batch, 10, axis=1)
        pred_upper_80[start_idx:end_idx] = np.percentile(expected_y_batch, 90, axis=1)
        pred_lower_68[start_idx:end_idx] = np.percentile(expected_y_batch, 16, axis=1)
        pred_upper_68[start_idx:end_idx] = np.percentile(expected_y_batch, 84, axis=1)
        pred_sd[start_idx:end_idx] = np.std(expected_y_batch, axis=1)

        # summary stats for each category probability
        for k in range(K):
            prob_cats_mean[start_idx:end_idx, k] = np.mean(prob_batch[:, :, k], axis=1)
            prob_cats_median[start_idx:end_idx, k] = np.median(prob_batch[:, :, k], axis=1)
            prob_cats_lower_95[start_idx:end_idx, k] = np.percentile(prob_batch[:, :, k], 2.5, axis=1)
            prob_cats_upper_95[start_idx:end_idx, k] = np.percentile(prob_batch[:, :, k], 97.5, axis=1)
            prob_cats_lower_90[start_idx:end_idx, k] = np.percentile(prob_batch[:, :, k], 5, axis=1)
            prob_cats_upper_90[start_idx:end_idx, k] = np.percentile(prob_batch[:, :, k], 95, axis=1)
            prob_cats_lower_80[start_idx:end_idx, k] = np.percentile(prob_batch[:, :, k], 10, axis=1)
            prob_cats_upper_80[start_idx:end_idx, k] = np.percentile(prob_batch[:, :, k], 90, axis=1)
            prob_cats_lower_68[start_idx:end_idx, k] = np.percentile(prob_batch[:, :, k], 16, axis=1)
            prob_cats_upper_68[start_idx:end_idx, k] = np.percentile(prob_batch[:, :, k], 84, axis=1)
            prob_cats_sd[start_idx:end_idx, k] = np.std(prob_batch[:, :, k], axis=1)

        all_draws[start_idx:end_idx, :] = expected_y_batch
        prob_all_draws[start_idx:end_idx, :, :] = prob_batch

        # Free batch arrays before next loop
        del mu_batch, prob_batch, expected_y_batch

    # Create compact outputs with stable row key for later merging
    cell_id = np.arange(n_cells, dtype=int)

    prediction_summary_df = pd.DataFrame({
        'cell_id': cell_id,
        f'{belief_col}_pred_mean': pred_mean,
        f'{belief_col}_pred_median': pred_median,
        f'{belief_col}_pred_lower_95': pred_lower_95,
        f'{belief_col}_pred_upper_95': pred_upper_95,
        f'{belief_col}_pred_lower_90': pred_lower_90,
        f'{belief_col}_pred_upper_90': pred_upper_90,
        f'{belief_col}_pred_lower_80': pred_lower_80,
        f'{belief_col}_pred_upper_80': pred_upper_80,
        f'{belief_col}_pred_lower_68': pred_lower_68,
        f'{belief_col}_pred_upper_68': pred_upper_68,
        f'{belief_col}_pred_sd': pred_sd,
    })

    for k in range(K):
        prediction_summary_df[f'{belief_col}_prob_cat_mean_{k+1}'] = prob_cats_mean[:, k]
        prediction_summary_df[f'{belief_col}_prob_cat_median_{k+1}'] = prob_cats_median[:, k]
        prediction_summary_df[f'{belief_col}_prob_cat_lower_95_{k+1}'] = prob_cats_lower_95[:, k]
        prediction_summary_df[f'{belief_col}_prob_cat_upper_95_{k+1}'] = prob_cats_upper_95[:, k]
        prediction_summary_df[f'{belief_col}_prob_cat_lower_90_{k+1}'] = prob_cats_lower_90[:, k]
        prediction_summary_df[f'{belief_col}_prob_cat_upper_90_{k+1}'] = prob_cats_upper_90[:, k]
        prediction_summary_df[f'{belief_col}_prob_cat_lower_80_{k+1}'] = prob_cats_lower_80[:, k]
        prediction_summary_df[f'{belief_col}_prob_cat_upper_80_{k+1}'] = prob_cats_upper_80[:, k]
        prediction_summary_df[f'{belief_col}_prob_cat_lower_68_{k+1}'] = prob_cats_lower_68[:, k]
        prediction_summary_df[f'{belief_col}_prob_cat_upper_68_{k+1}'] = prob_cats_upper_68[:, k]
        prediction_summary_df[f'{belief_col}_prob_cat_sd_{k+1}'] = prob_cats_sd[:, k]

    expected_value_draws_df = pd.DataFrame(
        all_draws,
        columns=[f'draw_{i+1}' for i in range(n_draws)]
    )
    expected_value_draws_df.insert(0, 'cell_id', cell_id)

    prob_draw_cols = []
    prob_draw_blocks = []
    for k in range(K):
        cols_k = [f'prob_cat_{k+1}_draw_{i+1}' for i in range(n_draws)]
        prob_draw_cols.extend(cols_k)
        prob_draw_blocks.append(prob_all_draws[:, :, k])

    prob_category_draws_df = pd.DataFrame(
        np.concatenate(prob_draw_blocks, axis=1),
        columns=prob_draw_cols
    )
    prob_category_draws_df.insert(0, 'cell_id', cell_id)

    return prediction_summary_df, expected_value_draws_df, prob_category_draws_df, K, draw_indices

# prediction_summary_df: summary stats only (cell_id + expected value + category probability summaries)
# expected_value_draws_df: all expected value draws by cell_id
# prob_category_draws_df: all category probability draws by cell_id
# K: number of belief categories
# draw_indices: selected posterior draw indices

In [27]:
# weighted group means and summaries

def aggregate_draws_by_group(draws_df, group_col):
    """
    Aggregate expected value draws by a single group column with posterior weight draws.
    - For each weight draw, compute weighted aggregate of belief draws
    - Average the aggregated results across all weight draws
    """
    draw_cols = [c for c in draws_df.columns if c.startswith('draw_')]
    if not draw_cols:
        raise ValueError("No draw_* columns found in draws_df.")

    # Check for posterior weight draws (N_star.1, N_star.2, ..., N_star.104)
    n_star_cols = [c for c in draws_df.columns if c.startswith('N_star.')]
    n_star_draws = sorted([c for c in n_star_cols if re.match(r'N_star\.\d+$', c)])
    
    print(f"Using {len(n_star_draws)} posterior weight draws for aggregation")
    all_results = []
        
    weight_indices = np.random.choice(len(n_star_draws), size = len(draw_cols))
    selected_weights = [n_star_draws[idx] for idx in weight_indices]

    for draw_col, weight_col in zip(draw_cols, selected_weights):
        w = draws_df[weight_col].astype(float)
        weighted_draw = draws_df[draw_col] * w
        num = weighted_draw.groupby(draws_df[group_col]).sum() #the sum for the expected belief value in the demographic group
        den = w.groupby(draws_df[group_col]).sum() # the sum of weights in one demographic group
        group_result = (num/den).reset_index()
        group_result.columns = [group_col, draw_col]
        group_result.set_index(group_col, inplace=True)  # Set group_col as index for proper concatenation
        all_results.append(group_result)
    
    # Concatenate along axis=1 and reset index to get the group column back
    group_draws = pd.concat(all_results, axis=1).reset_index()

    return group_draws #(row = n_groups, columns = group_col + draw_cols)

def summarize_group_draws(group_draws_df, group_col):
    draw_cols = [c for c in group_draws_df.columns if c.startswith('draw_')]
    vals = group_draws_df[draw_cols]
    out = pd.DataFrame({group_col: group_draws_df[group_col]})
    out['pred_mean'] = vals.mean(axis=1)
    out['pred_median'] = vals.median(axis=1)
    out['pred_lower_95'] = vals.quantile(0.025, axis=1)
    out['pred_upper_95'] = vals.quantile(0.975, axis=1)
    out['pred_lower_90'] = vals.quantile(0.05, axis=1)
    out['pred_upper_90'] = vals.quantile(0.95, axis=1)
    out['pred_lower_80'] = vals.quantile(0.10, axis=1)
    out['pred_upper_80'] = vals.quantile(0.90, axis=1)
    out['pred_lower_68'] = vals.quantile(0.16, axis=1)
    out['pred_upper_68'] = vals.quantile(0.84, axis=1)
    out['pred_sd'] = vals.std(axis=1)
    return out # (row = n_groups, columns = group_col + 5 summary stats)

In [36]:
# Check LP dimensions
base_dir = f"/mnt/e/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/{belief_col}"
# Find all time window directories
time_window_dirs = [d for d in os.listdir(base_dir) if d.startswith(belief_col)]
time_windows = sorted([d.replace(f"{belief_col}_", "").replace("_stan_output", "") for d in time_window_dirs])
for idx, window in enumerate(time_windows):
    print(f"[{idx+1}/{len(time_windows)}] Processing predictions for window: {window}")
    
    # Load the fitted model for this window
    model_dir = f"{base_dir}/{belief_col}_{window}_stan_output"
    fit_path = f"{model_dir}/fit.pkl"
    with open(fit_path, "rb") as f:
                fit = pickle.load(f)
    mu = fit.stan_variable("mu_state_LP")   # (n_draws, N_states)
    beta = fit.stan_variable("beta_LP")     # (n_draws, 3)
    LP = LP_mat                              # (N_states, 3)

    # contributions from each LP column to mu_state_LP
    # result shape: (n_draws, N_states, 3)
    contrib = beta[:, None, :] * LP[None, :, :]

    print("mu_state_LP shape:", mu.shape)
    print("beta_LP shape:", beta.shape)
    print("LP_mat shape:", LP.shape)
    print("contrib shape (draws, states, LPs):", contrib.shape)
    summary = fit.summary()
    print(summary.loc[[idx for idx in summary.index if idx.startswith("beta_LP[")]])
    # see first draw, first 5 states, all 3 LP contributions
    print(contrib[0, :5, :])
    # compare eta_state and mu_state which one contribute more
    eta_state = fit.stan_variable("eta_state")  # (draws, states)
    mu_state_LP = fit.stan_variable("mu_state_LP")

    print("SD of mu_state_LP:", np.std(mu_state_LP))
    print("SD of eta_state:", np.std(eta_state))
    # contrib shape is (draws, states, LPs)
    avg_abs = np.mean(np.abs(contrib), axis=(0, 1))
    sd_contrib = np.std(contrib, axis=(0, 1))

    impact_df = pd.DataFrame({
        "LP": LP_cols,
        "mean_abs_contribution": avg_abs,
        "sd_contribution": sd_contrib
    })
    print(impact_df)
    break


[1/26] Processing predictions for window: 2023-11-05 00:00:00+00:00
mu_state_LP shape: (256, 51)
beta_LP shape: (256, 3)
LP_mat shape: (51, 3)
contrib shape (draws, states, LPs): (256, 51, 3)
                Mean      MCSE    StdDev       MAD        5%       50%  \
beta_LP[1] -0.005372  0.022697  0.381589  0.354743 -0.673069  0.001761   
beta_LP[2] -0.011234  0.011686  0.206687  0.228847 -0.340773 -0.003510   
beta_LP[3] -0.242932  0.022511  0.355961  0.326077 -0.809710 -0.237011   

                 95%  ESS_bulk  ESS_tail  ESS_bulk/s     R_hat  
beta_LP[1]  0.582038   306.838     128.0    0.335004  1.008140  
beta_LP[2]  0.282679   332.477     128.0    0.362996  0.996647  
beta_LP[3]  0.360876   267.511     128.0    0.292067  1.025270  
[[-8.67385383e-02 -3.56277886e-02 -3.09688269e-04]
 [-2.37947046e-01  2.13140151e-01 -1.43504099e-01]
 [-2.59145204e-01  4.30884066e-01 -5.95625478e-02]
 [ 1.50777055e-02 -3.11559704e-02  3.21726766e-03]
 [ 2.98384009e-01 -1.40860310e-02  6.76335907e-

In [ ]:
import pickle
import pandas as pd
import numpy as np
import os

# compute group-level category probabilities from cell-level predictions
def compute_group_category_probabilities(pred_df, belief_col, group_col):
    """
    - For each weight draw, compute weighted group category probabilities
    - Average results across all 104 weight draws
    """
    cat_cols = [c for c in pred_df.columns if c.startswith(f"{belief_col}_prob_cat_mean_")]
    K = max(sorted(int(c.split('_')[-1]) for c in cat_cols)) #number of categories

    n_star_cols = [c for c in pred_df.columns if c.startswith('N_star.')]
    n_star_draws = sorted([c for c in n_star_cols if re.match(r'N_star\.\d+$', c)])
    
    # Randomly sample one weight for each category
    weight_indices = np.random.choice(len(n_star_draws), size=K)
    selected_weights = [n_star_draws[idx] for idx in weight_indices]
    
    # Compute weighted means per category using randomly selected weights
    out_rows = []
    for k in range(1, K+1):
        col_k = f"{belief_col}_prob_cat_mean_{k}"
        if col_k not in pred_df.columns:
            continue
        
        # Use randomly selected weight for this category
        w_col = selected_weights[k-1]
        w = pred_df[w_col].astype(float)
        
        # Compute weighted aggregation
        num_by_group = (pred_df[col_k] * w).groupby(pred_df[group_col]).sum()
        denom_by_group = w.groupby(pred_df[group_col]).sum()
        prob_by_group = (num_by_group / denom_by_group).reset_index()
        prob_by_group.columns = [group_col, 'prob_mean']
        prob_by_group['category'] = k
        out_rows.append(prob_by_group)
    
    if not out_rows:
        raise ValueError(f"No results generated during aggregation. Check that belief column '{belief_col}' and weight columns are present in pred_df.")
    
    final_result = pd.concat(out_rows, ignore_index=True)
    
    return final_result # (row = n_groups * K, columns = group_col, category, prob_mean)

strat_frame = pd.read_csv("/mnt/e/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/feature_extraction/state_LP_cleaned_sf.csv")

# Get the time windows from fitted models
base_dir = f"/mnt/e/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/{belief_col}"

# Find all time window directories
time_window_dirs = [d for d in os.listdir(base_dir) if d.startswith(belief_col)]
time_windows = sorted([d.replace(f"{belief_col}_", "").replace("_stan_output", "") for d in time_window_dirs])

print(f"Found {len(time_windows)} time windows: {time_windows}\n")

# Group mapping for consolidation
group_map = [
    ('state_simple', 'state'),
    ('state_idx', 'state_idx'),
    ('sex_idx', 'sex'),
    ('age_idx', 'age'),
    ('eth_idx', 'eth'),
    ('income_idx', 'income'),
    ('vote_idx', 'vote'),
    ('education_idx', 'education')
]

# Initialize storage for consolidated category probabilities across all windows
consolidated_cat_probs = {name: [] for _, name in group_map}

# columns needed only for downstream aggregation / weighting
n_star_cols = [c for c in strat_frame.columns if c.startswith('N_star.')]
n_star_draws = sorted([c for c in n_star_cols if re.match(r'N_star\.\d+$', c)])
merge_cols = ['state_simple', 'state_idx', 'sex_idx', 'age_idx', 'eth_idx', 'income_idx', 'vote_idx', 'education_idx']
merge_cols = [c for c in merge_cols if c in strat_frame.columns]
meta_cols = ['N_star.median'] + merge_cols if 'N_star.median' in strat_frame.columns else merge_cols
meta_cols = n_star_draws + meta_cols

# Loop over each time window
for idx, window in enumerate(time_windows):
    print(f"[{idx+1}/{len(time_windows)}] Processing predictions for window: {window}")
    
    # Load the fitted model for this window
    model_dir = f"{base_dir}/{belief_col}_{window}_stan_output"
    fit_path = f"{model_dir}/fit.pkl"
    
    if not os.path.exists(fit_path):
        print(f"  Warning: fit.pkl not found for window {window}")
        continue
    
    try:
        with open(fit_path, "rb") as f:
            fit = pickle.load(f)
        
        # Generate cell-level predictions (update: no copied stratification columns)
        strat_frame_copy = strat_frame.copy()
        prediction_summary_df, expected_value_draws_df, prob_category_draws_df, K, draw_indices = generate_cell_level_predictions(
            fit,
            strat_frame_copy,
            belief_col,
            batch_size=10000,
            n_keep_draws=250
        )

        # stratification rows must exactly match generated cell-level rows
        n_cells_expected = len(strat_frame_copy)
        if (
            len(prediction_summary_df) != n_cells_expected
            or len(expected_value_draws_df) != n_cells_expected
            or len(prob_category_draws_df) != n_cells_expected
        ):
            warning_msg = (
                "WARNING: Length mismatch detected between stratification frame and cell-level outputs. "
                "Stopping execution with no fallback."
            )
            print(warning_msg)
            raise ValueError(warning_msg)

        # Save compact cell-level outputs
        prediction_summary_df.to_csv(f"{model_dir}/cell_level_predictions.csv", index=False)
        # expected_value_draws_df.to_csv(f"{model_dir}/cell_level_expected_value_draws.csv", index=False)
        # prob_category_draws_df.to_csv(f"{model_dir}/cell_level_category_probability_draws.csv.gz", index=False, compression='gzip')

        # Save selected draw indices
        pd.Series(draw_indices, name='draw_index').to_csv(f"{model_dir}/selected_draw_indices.csv", index=False)

        # # Save kappa draws corresponding to selected draw indices
        # try:
        #     kappa_full = fit.stan_variable('kappa')
        #     kappa_sub = kappa_full[draw_indices, :]
        #     kappa_sub_df = pd.DataFrame(kappa_sub, columns=[f'kappa_{i+1}' for i in range(kappa_sub.shape[1])])
        #     kappa_sub_df.to_csv(f"{model_dir}/kappa_subsampled.csv", index=False)
        # except Exception as e:
        #     print(f"Warning: failed to save kappa subsample: {e}")

        # Build merge-ready metadata frame (stable key + original order)
        merge_meta = strat_frame_copy.reset_index(drop=True).loc[:, meta_cols].copy()
        merge_meta.insert(0, 'cell_id', np.arange(len(merge_meta), dtype=int))

        # Temporary frames for weighted aggregation only
        draws_for_agg = expected_value_draws_df.merge(merge_meta, on='cell_id', how='left')
        pred_for_agg = prediction_summary_df.merge(merge_meta, on='cell_id', how='left')

        # State-level aggregation and save (expected value draws)
        state_group_col = 'state_simple' if 'state_simple' in draws_for_agg.columns else ('state_idx' if 'state_idx' in draws_for_agg.columns else None)
        if state_group_col is not None:
            state_all_draws = aggregate_draws_by_group(draws_for_agg, state_group_col)
            state_preds = summarize_group_draws(state_all_draws, state_group_col)
            state_all_draws.to_csv(f"{model_dir}/group_state_all_draws.csv", index=False)
            state_preds.to_csv(f"{model_dir}/group_state_predictions.csv", index=False)
        else:
            print("Warning: no state column found; skipping state-level aggregation.")

        # Demographic group aggregations and save (expected value draws)
        demo_group_map = [
            ('sex_idx', 'sex'),
            ('age_idx', 'age'),
            ('eth_idx', 'eth'),
            ('income_idx', 'income'),
            ('vote_idx', 'vote'),
            ('education_idx', 'education')
        ]
        for gcol, gname in demo_group_map:
            if gcol in draws_for_agg.columns:
                grp_draws = aggregate_draws_by_group(draws_for_agg, gcol)
                grp_preds = summarize_group_draws(grp_draws, gcol)
                grp_draws.to_csv(f"{model_dir}/group_{gname}_all_draws.csv", index=False)
                grp_preds.to_csv(f"{model_dir}/group_{gname}_predictions.csv", index=False)
            else:
                print(f"Note: column '{gcol}' not found; skipping group '{gname}'.")
        
        # Compute category probabilities for all groups and store for consolidation
        print(f"  Computing category probabilities using all 104 weight samples...")
        for gcol, gname in group_map:
            if gcol not in pred_for_agg.columns:
                continue
            try:
                grp_cat = compute_group_category_probabilities(
                    pred_for_agg, belief_col, gcol
                )
                grp_cat['time_window_start'] = window
                consolidated_cat_probs[gname].append(grp_cat)
            except Exception as e:
                print(f"    Warning: failed to compute {gname} category probs: {e}")
        
        print(f"  Successfully generated all predictions for window {window}\n")
        
    except Exception as e:
        if "Length mismatch detected between stratification frame and cell-level outputs" in str(e):
            raise
        print(f"  Failed to process window {window}: {e}\n")
        continue

print("All time windows processed!")

# Save consolidated category probability timeseries files
print("\nSaving consolidated category probability timeseries...")
for gname, parts in consolidated_cat_probs.items():
    if parts:
        df_all = pd.concat(parts, ignore_index=True)
        output_path = f"{base_dir}/group_{gname}_category_probs_timeseries.csv"
        df_all.to_csv(output_path, index=False)
        print(f" Saved {gname}: {df_all.shape[0]} rows, {df_all['time_window_start'].nunique()} windows")
    else:
        print(f" No data for {gname}")

print("\nPrediction and consolidation pipeline complete!")

Found 26 time windows: ['2023-11-05 00:00:00+00:00', '2023-11-19 00:00:00+00:00', '2023-12-03 00:00:00+00:00', '2023-12-17 00:00:00+00:00', '2023-12-31 00:00:00+00:00', '2024-01-14 00:00:00+00:00', '2024-01-28 00:00:00+00:00', '2024-02-11 00:00:00+00:00', '2024-02-25 00:00:00+00:00', '2024-03-10 00:00:00+00:00', '2024-03-24 00:00:00+00:00', '2024-04-07 00:00:00+00:00', '2024-04-21 00:00:00+00:00', '2024-05-05 00:00:00+00:00', '2024-05-19 00:00:00+00:00', '2024-06-02 00:00:00+00:00', '2024-06-16 00:00:00+00:00', '2024-06-30 00:00:00+00:00', '2024-07-14 00:00:00+00:00', '2024-07-28 00:00:00+00:00', '2024-08-11 00:00:00+00:00', '2024-08-25 00:00:00+00:00', '2024-09-08 00:00:00+00:00', '2024-09-22 00:00:00+00:00', '2024-10-06 00:00:00+00:00', '2024-10-20 00:00:00+00:00']

[1/26] Processing predictions for window: 2023-11-05 00:00:00+00:00
Subsampling draws: using 250 out of 256 posterior draws

Generating predictions for 190,844 cells in 20 batches of size 10000
Using 104 posterior weight 